# Public vs Private Collections — customer demo

A two-scenario walkthrough that mirrors the real customer use cases:

- **Scenario A — Public collection**: writes go through the catalog API, items land in PostgreSQL, the OUTBOX drains them into the shared `dynastore-items` Elasticsearch alias. Anyone (including anonymous callers) can search them via STAC.
- **Scenario B — Private collection**: same write path, but the collection's routing configs pin the private driver variants (`items_elasticsearch_private_driver` + `collection_elasticsearch_private_driver`). Items are still indexed for authenticated search, but anonymous STAC search must return zero results and never leak content.

Both scenarios run end-to-end against the live review environment.

`DYNASTORE_TOKEN` is required for write operations. The script reads it from the environment (or `.env` via `python-dotenv` when available).

In [ ]:
import json
import os
import time
import uuid

import httpx

try:
    from dotenv import load_dotenv, find_dotenv  # type: ignore
    load_dotenv(find_dotenv(usecwd=True), override=False)
except Exception:
    pass

BASE_URL = os.environ.get(

# Auto-provision DYNASTORE_TOKEN via IDP client_credentials if not already set
if not os.environ.get("DYNASTORE_TOKEN"):
    _idp = (os.environ.get("IDP_PUBLIC_URL") or os.environ.get("IDP_ISSUER_URL", "")).rstrip("/")
    _cid = os.environ.get("IDP_CLIENT_ID", "")
    _sec = os.environ.get("IDP_CLIENT_SECRET", "")
    if _idp and _cid and _sec:
        try:
            _r = httpx.post(
                f"{_idp}/protocol/openid-connect/token",
                data={"grant_type": "client_credentials", "client_id": _cid, "client_secret": _sec},
                timeout=10,
            )
            if _r.status_code == 200:
                os.environ["DYNASTORE_TOKEN"] = _r.json().get("access_token", "")
        except Exception:
            pass
    "DYNASTORE_BASE_URL",
    "http://localhost:8080",
)
ANON_BASE_URL = os.environ.get(
    "DYNASTORE_ANON_BASE_URL",
    "http://localhost:8080",
)
TOKEN = os.environ.get("DYNASTORE_TOKEN") or os.environ.get("ACCESS_TOKEN")
if not TOKEN:
    raise RuntimeError("Set DYNASTORE_TOKEN to a valid bearer token before running.")

RUN_ID = uuid.uuid4().hex[:6]
PUB_CATALOG_ID = f"demo-public-{RUN_ID}"
PUB_COLL_ID = f"public-coll-{RUN_ID}"
PRIV_CATALOG_ID = f"demo-private-{RUN_ID}"
PRIV_COLL_ID = f"private-coll-{RUN_ID}"

DEFAULT_HEADERS = {"Content-Type": "application/json", "Accept": "application/json"}
AUTH_HEADERS = {**DEFAULT_HEADERS, "Authorization": f"Bearer {TOKEN}"}

client = httpx.Client(base_url=BASE_URL, headers=AUTH_HEADERS, timeout=30.0)
anon = httpx.Client(base_url=ANON_BASE_URL, headers=DEFAULT_HEADERS, timeout=30.0)

print(f"RUN_ID={RUN_ID}\n  authenticated base: {BASE_URL}\n  anonymous base:     {ANON_BASE_URL}")


## Helpers — readiness polls

Two small helpers used throughout the demo:

- `wait_for_catalog(cat_id)` — tenant provisioning is async, so a request issued in the same millisecond as the catalog POST can still 404. Poll `GET /stac/catalogs/{cat}` until it answers (200/401/403 all prove the tenant exists).
- `wait_for_search(cat_id)` — items go through PostgreSQL → OUTBOX → Elasticsearch. Search returns nothing until the drain completes; poll the search endpoint instead of guessing a fixed sleep.

In [ ]:
def wait_for_catalog(cat_id, timeout_s=30, poll_s=1.0):
    """GET on the catalog returns 200/401/403 once the schema exists. Returns last (status, body)."""
    deadline = time.monotonic() + timeout_s
    last = (None, "")
    while time.monotonic() < deadline:
        r = client.get(f"/stac/catalogs/{cat_id}")
        last = (r.status_code, r.text[:300])
        if r.status_code in (200, 401, 403):
            return last
        time.sleep(poll_s)
    raise TimeoutError(f"catalog {cat_id} not reachable in {timeout_s}s; last={last}")

def create_collection_when_ready(cat_id, body, timeout_s=120):
    """POST a collection, retrying on 409 while the catalog tenant is still provisioning.

    Catalog GET returns 200 as soon as the row is committed, but the schema/extension
    plumbing (STAC sidecar, ES alias add, write-policy install) finishes asynchronously.
    During that window collection POST can return 409 even though there is no genuine
    duplicate. We retry with exponential backoff and surface the last response on timeout.

    Same shape applies to records / features / coverages / assets create endpoints —
    use the same wrapper for any OGC create-on-fresh-catalog flow.
    """
    path = f"/stac/catalogs/{cat_id}/collections"
    deadline = time.monotonic() + timeout_s
    delay = 1.0
    attempt = 0
    last = None
    while time.monotonic() < deadline:
        attempt += 1
        r = client.post(path, content=json.dumps(body))
        last = r
        if r.is_success:
            print(f"  collection {body['id']!r}: {r.status_code} (attempt {attempt})")
            return r
        if r.status_code == 409:
            print(f"  collection {body['id']!r}: 409 (attempt {attempt}) — catalog still provisioning, retry in {delay:.1f}s")
            time.sleep(delay)
            delay = min(delay * 1.7, 8.0)
            continue
        raise RuntimeError(f"collection POST failed: {r.status_code} {r.text[:300]}")
    raise TimeoutError(
        f"collection {body['id']!r}: still 409 after {timeout_s}s; last={last.status_code if last else None} {last.text[:300] if last else ''}"
    )

def wait_for_search(cat_id, expected, timeout_s=60, poll_s=2.0):
    """Poll authenticated STAC search until at least `expected` features land. Returns (count, status, body)."""
    deadline = time.monotonic() + timeout_s
    last_status, last_body, count = None, "", 0
    while time.monotonic() < deadline:
        r = client.post(f"/search/catalogs/{cat_id}", content=json.dumps({"limit": 50}))
        last_status, last_body = r.status_code, r.text[:300]
        if r.is_success:
            count = len(r.json().get("features", []))
            if count >= expected:
                return count, last_status, last_body
        time.sleep(poll_s)
    return count, last_status, last_body

def diagnose_search_miss(cat_id):
    """Layer-by-layer probe so a 0-feature outcome points at the actual broken layer.

    1. PG-hint fallback — confirms whether items reached the SQL store at all.
    2. Items WRITE driver introspect — which driver received the writes
       (PG vs ES vs both). If the WRITE driver is ES-only, items will never
       appear in PG.
    3. Index failures endpoint — only mounted in SCOPEs with the OUTBOX
       drain task; a 404 here means "endpoint not deployed in this SCOPE",
       not "no failures".
    """
    out = {}
    r = client.post(
        f"/search/catalogs/{cat_id}",
        params={"hint": "geometry_exact"},
        content=json.dumps({"limit": 50}),
    )
    out["pg_hint"] = {
        "status": r.status_code,
        "count": len(r.json().get("features", [])) if r.is_success else None,
        "body": r.text[:300],
    }
    r = client.get(f"/configs/catalogs/{cat_id}/items")
    out["write_driver"] = {"status": r.status_code, "body": r.text[:600]}
    r = client.get(f"/catalogs/{cat_id}/index-failures")
    if r.status_code == 404:
        out["index_failures"] = "endpoint not deployed in this SCOPE (no OUTBOX drain) — open audit bug #1"
    else:
        out["index_failures"] = {"status": r.status_code, "body": r.text[:600]}
    return out


---
## Scenario A — Public collection

Items land in the shared ES alias and are searchable by anonymous callers.

### A.1 — Create the catalog and wait for it to be reachable

In [ ]:
r = client.post("/stac/catalogs", content=json.dumps({
    "id": PUB_CATALOG_ID,
    "type": "Catalog", "stac_version": "1.0.0",
    "title": f"Demo public {RUN_ID}",
    "description": "Customer demo — safe to delete",
    "links": [],
}))
if r.status_code not in (200, 201, 409):
    raise RuntimeError(f"catalog create failed: {r.status_code} {r.text[:300]}")
print(f"Catalog {PUB_CATALOG_ID!r}: {r.status_code}")
wait_for_catalog(PUB_CATALOG_ID)
print(f"Catalog {PUB_CATALOG_ID!r} ready")

### A.2 — Create the public collection

In [ ]:
create_collection_when_ready(PUB_CATALOG_ID, {
    "id": PUB_COLL_ID, "type": "Collection", "stac_version": "1.0.0",
    "description": "Public items demo", "license": "proprietary",
    "extent": {"spatial": {"bbox": [[-180,-90,180,90]]}, "temporal": {"interval": [[None, None]]}},
    "links": [],
})


### A.3 — Ingest 3 public STAC items

In [ ]:
PUB_ITEM_IDS = []
for i in range(3):
    item_id = f"pub-item-{i}-{RUN_ID}"
    r = client.post(
        f"/stac/catalogs/{PUB_CATALOG_ID}/collections/{PUB_COLL_ID}/items",
        content=json.dumps({
            "type": "Feature", "stac_version": "1.0.0",
            "id": item_id,
            "geometry": {"type": "Point", "coordinates": [12.5 + i * 0.1, 41.9]},
            "bbox": [12.4 + i * 0.1, 41.8, 12.6 + i * 0.1, 42.0],
            "properties": {"datetime": "2024-06-01T00:00:00Z"},
            "links": [], "assets": {},
        }),
    )
    if r.status_code in (200, 201):
        PUB_ITEM_IDS.append(item_id)
        print(f"  item {item_id}: {r.status_code} ok")
    else:
        print(f"  item {item_id}: {r.status_code} {r.text[:200]}")

if len(PUB_ITEM_IDS) != 3:
    raise RuntimeError(f"only {len(PUB_ITEM_IDS)}/3 items ingested — see errors above")

### A.4 — Authenticated STAC search returns the items

Polls until the OUTBOX has drained the new items into Elasticsearch.

In [ ]:
count, status, body = wait_for_search(PUB_CATALOG_ID, expected=3, timeout_s=60)
print(f"authenticated search: HTTP {status}, {count} feature(s) after polling")
if count >= 3:
    print("  OK — ES routing + OUTBOX drain are working")
else:
    diag = diagnose_search_miss(PUB_CATALOG_ID)
    print(f"  diagnostic: {json.dumps(diag, indent=2)}")
    if diag["pg_hinted_count"] and diag["pg_hinted_count"] >= 3:
        raise RuntimeError(
            f"items present in PG ({diag['pg_hinted_count']}) but invisible via the "
            f"ES alias — check that {PUB_CATALOG_ID!r} is registered in dynastore-items."
        )
    raise RuntimeError(
        f"items missing from both ES ({count}) and PG ({diag['pg_hinted_count']}) — "
        f"see index-failures: {diag['index_failures']!r}"
    )

### A.5 — Anonymous STAC search also returns the items

Same query, no `Authorization` header. A public collection must be visible to everyone.

In [ ]:
r = anon.post(
    f"/search/catalogs/{PUB_CATALOG_ID}",
    content=json.dumps({"limit": 10})
)
print(f"anonymous search: HTTP {r.status_code}")
feats = r.json().get("features", []) if r.is_success else []
print(f"  features: {len(feats)}")
if len(feats) >= 3:
    print("  OK — public items visible without auth")
else:
    raise RuntimeError("public collection should be readable anonymously but returned 0 features")

---
## Scenario B — Private collection

Same code path; pinning the private driver variants in the collection's routing configs routes its items + envelope to per-tenant private indexes guarded by a catalog-wide DENY policy so anonymous callers see nothing.

### B.1 — Create the private catalog + collection

In [ ]:
r = client.post("/stac/catalogs", content=json.dumps({
    "id": PRIV_CATALOG_ID,
    "type": "Catalog", "stac_version": "1.0.0",
    "title": f"Demo private {RUN_ID}",
    "description": "Customer demo — private collection scenario",
    "links": [],
}))
print(f"POST /stac/catalogs (private): {r.status_code}")
if r.status_code not in (200, 201):
    raise RuntimeError(f"create private catalog failed: {r.status_code} {r.text[:300]}")
wait_for_catalog(PRIV_CATALOG_ID)
create_collection_when_ready(PRIV_CATALOG_ID, {
    "id": PRIV_COLL_ID, "type": "Collection", "stac_version": "1.0.0",
    "description": "Private items demo", "license": "proprietary",
    "extent": {"spatial": {"bbox": [[-180,-90,180,90]]}, "temporal": {"interval": [[None, None]]}},
    "links": [],
})


### B.2 — Make the collection private via routing configs

Privacy is expressed by pinning the private driver variants in the per-collection routing configs. Two PUTs:

1. `PUT /configs/catalogs/{cat}/collections/{coll}/plugins/items_routing_config` — items routing pinning `items_elasticsearch_private_driver`.
2. `PUT /configs/catalogs/{cat}/collections/{coll}/plugins/collection_routing_config` — collection-envelope routing pinning `collection_elasticsearch_private_driver`.

The cascade validator requires the items-private driver to be pinned first; the cell below writes them in that order.

In [ ]:
PRIV_ITEMS_ROUTING = {
    "operations": {
        "write": [
            {"driver_ref": "items_postgresql_driver", "on_failure": "fatal"},
            {"driver_ref": "items_elasticsearch_private_driver", "write_mode": "async", "on_failure": "outbox"},
        ],
        "read": [{"driver_ref": "items_postgresql_driver"}],
        "index": [
            {"driver_ref": "items_elasticsearch_private_driver", "write_mode": "async", "on_failure": "outbox"},
        ],
        "search": [{"driver_ref": "items_elasticsearch_private_driver"}],
    }
}
PRIV_COLLECTION_ROUTING = {
    "operations": {
        "write": [{"driver_ref": "collection_postgresql_driver", "on_failure": "fatal"}],
        "read":  [{"driver_ref": "collection_postgresql_driver", "on_failure": "fatal"}],
        "index": [{"driver_ref": "collection_elasticsearch_private_driver", "write_mode": "async", "on_failure": "outbox"}],
        "search":[{"driver_ref": "collection_elasticsearch_private_driver"}],
    }
}

# Cascade rule: items-private must land BEFORE collection-private.
r = client.put(
    f"/configs/catalogs/{PRIV_CATALOG_ID}/collections/{PRIV_COLL_ID}/plugins/items_routing_config",
    content=json.dumps(PRIV_ITEMS_ROUTING),
)
if not r.is_success:
    raise RuntimeError(f"items routing write failed: {r.status_code} {r.text[:300]}")
print(f"PUT items_routing_config: {r.status_code}")

r = client.put(
    f"/configs/catalogs/{PRIV_CATALOG_ID}/collections/{PRIV_COLL_ID}/plugins/collection_routing_config",
    content=json.dumps(PRIV_COLLECTION_ROUTING),
)
if not r.is_success:
    raise RuntimeError(f"collection routing write failed: {r.status_code} {r.text[:300]}")
print(f"PUT collection_routing_config: {r.status_code}")

### B.3 — Ingest 3 private items

In [ ]:
PRIV_ITEM_IDS = []
for i in range(3):
    item_id = f"priv-item-{i}-{RUN_ID}"
    r = client.post(
        f"/stac/catalogs/{PRIV_CATALOG_ID}/collections/{PRIV_COLL_ID}/items",
        content=json.dumps({
            "type": "Feature", "stac_version": "1.0.0",
            "id": item_id,
            "geometry": {"type": "Point", "coordinates": [13.5 + i * 0.1, 42.9]},
            "bbox": [13.4 + i * 0.1, 42.8, 13.6 + i * 0.1, 43.0],
            "properties": {"datetime": "2024-06-01T00:00:00Z", "confidential": True},
            "links": [], "assets": {},
        }),
    )
    if r.status_code in (200, 201):
        PRIV_ITEM_IDS.append(item_id)
        print(f"  item {item_id}: {r.status_code} ok")
    else:
        print(f"  item {item_id}: {r.status_code} {r.text[:200]}")

if len(PRIV_ITEM_IDS) != 3:
    raise RuntimeError(f"only {len(PRIV_ITEM_IDS)}/3 private items ingested")

### B.4 — Authenticated user sees the private items

In [ ]:
count, status, body = wait_for_search(PRIV_CATALOG_ID, expected=3, timeout_s=60)
print(f"authenticated search: HTTP {status}, {count} feature(s)")
if count < 3:
    diag = diagnose_search_miss(PRIV_CATALOG_ID)
    print(f"  diagnostic: {json.dumps(diag, indent=2)}")
    raise RuntimeError("authenticated user should see all 3 private items")

### B.5 — Anonymous caller sees nothing

The same query without an `Authorization` header must return either 401/403 or 0 features. A non-empty result here would be a privacy leak and the cell will raise.

In [ ]:
r = anon.post(
    f"/search/catalogs/{PRIV_CATALOG_ID}",
    content=json.dumps({"limit": 10})
)
print(f"anonymous search: HTTP {r.status_code}")
if r.status_code in (401, 403):
    print("  OK — anonymous access rejected at the auth layer")
elif r.is_success and not r.json().get("features", []):
    print("  OK — anonymous search returned 0 features (privacy enforced)")
else:
    feats = r.json().get("features", []) if r.is_success else []
    raise RuntimeError(
        f"PRIVACY LEAK: anonymous caller got {len(feats)} private feature(s); "
        f"first id: {feats[0].get('id') if feats else None}"
    )

---
## Recap

| Scenario | Authenticated | Anonymous |
|---|---|---|
| Public collection — STAC search | OK (A.4) | OK (A.5) |
| Private collection — STAC search | OK (B.4) | DENY / 0 features (B.5) |

The same write path serves both — the only difference is the routing-config writes (items + collection) in B.2.

## Teardown

In [ ]:
for cat_id in [PUB_CATALOG_ID, PRIV_CATALOG_ID]:
    r = client.delete(f"/stac/catalogs/{cat_id}", params={"force": "true"})
    print(f"DELETE {cat_id!r}: {r.status_code}")
client.close()
anon.close()
print("Done.")